### COT Report data scraper

https://github.com/NDelventhal/cot_reports

In [52]:
import pandas as pd
import cot_reports as cot
from matplotlib import pyplot as plt
import numpy as np
import re
#%matplotlib widget
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML


### <u> 1. Instructions for using cot_reports </u>


From https://github.com/NDelventhal/cot_reports

#### Example: cot_hist()
df = cot.cot_hist(cot_report_type= 'traders_in_financial_futures_futopt')
#### cot_hist() downloads the historical bulk file for the specified report type, in this example the Traders in Financial Futures Futures-and-Options Combined report. Returns the data as dataframe.

#### Example: cot_year()
df = cot.cot_year(year = 2020, cot_report_type = 'traders_in_financial_futures_fut')
#### cot_year() downloads the single year file of the specified report type and year. Returns the data as dataframe.

#### Example for collecting data of a few years, here from 2017 to 2020, of a specified report:
df = pd.DataFrame()
begin_year = 2017
end_year = 2020
for i in range(begin_year, end_year + 1):
    single_year = pd.DataFrame(cot.cot_year(i, cot_report_type='legacy_futopt')) 
    df = df.append(single_year, ignore_index=True)

#### Example: cot_all()
df = cot.cot_all(cot_report_type='legacy_fut')
#### cot_all() downloads the historical bulk file and all remaining single year files of the specified report type.  Returns the data as dataframe.

### 2. Download and Compile COT Data into a pandas dataframe

In [53]:
def cot_reader (start, end):
    df_list = []
    begin_year = start
    end_year = end
    for i in range(begin_year, end_year + 1):
        single_year = pd.DataFrame(cot.cot_year(i, cot_report_type='legacy_fut')) 
        df_list.append(single_year)
    
    df = pd.concat(df_list, ignore_index=True)
    
    df.rename(columns = {"Market and Exchange Names" : "Market", 
                     "As of Date in Form YYMMDD" : "Datetime", 
                     "Open Interest (All)" : "OI", 
                     "As of Date in Form YYYY-MM-DD" : "Date"}, inplace=True )
    
    df["Datetime"] = pd.to_datetime(df["Datetime"], format = '%y%m%d')
    
    df.sort_values("Datetime", ascending = False, inplace = True)
    
    return df

In [54]:
df = cot_reader(2022, 2026)

Selected: legacy_fut
Downloaded single year data from: 2022
Stored the file annual.txt in the working directory.
Selected: legacy_fut
Downloaded single year data from: 2023
Stored the file annual.txt in the working directory.
Selected: legacy_fut
Downloaded single year data from: 2024
Stored the file annual.txt in the working directory.
Selected: legacy_fut
Downloaded single year data from: 2025
Stored the file annual.txt in the working directory.
Selected: legacy_fut
Downloaded single year data from: 2026
Stored the file annual.txt in the working directory.


### 3. Create a list of markets to trade 

##### This is based on personal preference, please ignore if you're looking at all markets or edit as you like

In [55]:
keywords = ["gold", "silver", "platinum", "palladium", "copper", "Lithium", "crude", "heating", "oil", "Nat Gas",
"rbob", "brent", "cocoa", "corn", "oat", "wheat", "soybean", "soy bean", "feed" , "Hogs", "live", "OJ", "coffee", "cotton", "sugar","E-Mini", "Micro","WTI-PHYSICAL",
    "Russell",
    "S&P 500",
    "NASDAQ",
    "Dow Jones",
    "Nikkei",
    "FTSE",
    "DAX",
    "CAC",
    "SMI",
    "Hang Seng",
    "Shanghai",
    "Treasury", "UST", "Bond", "EURO" , "Peso", "Brazilian", "Swiss", "Canadian", "British", "Japanese", "New Zealand", "Rand", 
    "bitcoin" , "ether","SOFR", "vix"
           ]

In [56]:
unique_markets = []

for keyword in keywords:
    filtered_df = df[df["Market"].str.contains(keyword, case=False)]
    
    unique_values = filtered_df["Market"].unique()
    
    unique_markets.extend(unique_values)
    

In [57]:
remove_items = [
    'EURODOLLARS-3M - CHICAGO MERCANTILE EXCHANGE',
    'EURO SHORT TERM RATE - CHICAGO MERCANTILE EXCHANGE',
    'ALUMINIUM EURO PREM DUTY-PAID - COMMODITY EXCHANGE INC.',
    'EURODOLLARS-3M - CHICAGO MERCANTILE EXCHANGE',
    '3-MONTH EURODOLLARS - CHICAGO MERCANTILE EXCHANGE',
    'COFFEE CALENDAR SPREAD OPTIONS - ICE FUTURES U.S.',
    'WHEAT-HRW - CHICAGO BOARD OF TRADE',
    'WHEAT-HRSpring - MINNEAPOLIS GRAIN EXCHANGE',
    'BLACK SEA WHEAT FINANCIAL - CHICAGO BOARD OF TRADE',
    'CORN CONSECUTIVE CSO - CHICAGO BOARD OF TRADE',
    'CORN CSO - CHICAGO BOARD OF TRADE',
    'MARINE .5% FOB USGC/BRENT 1st - ICE FUTURES ENERGY DIV',
    'GASOLINE CRK-RBOB/BRENT 1st - ICE FUTURES ENERGY DIV',
    'WTI-BRENT SPREAD OPTION - NEW YORK MERCANTILE EXCHANGE',
    'WTI-BRENT CALENDAR - NEW YORK MERCANTILE EXCHANGE',
    'USGC HSFO-PLATTS/BRENT 1ST LN - ICE FUTURES ENERGY DIV',
    'TRANSCONTINENTAL GAS- STATION 85 (ZONE 4) - ICE FUTURES ENERGY DIV',
    'NORTHERN NATURAL GAS - VENTURA (BASIS) - ICE FUTURES ENERGY DIV',
    'NATURAL GAS PIPELINE-TEXOK (BASIS) - ICE FUTURES ENERGY DIV',
    'PANHANDLE EASTERN- POOL GAS (BASIS) - ICE FUTURES ENERGY DIV',
    'PACIFIC GAS TRANSMISSION - MALIN (BASIS) - ICE FUTURES ENERGY DIV',
    'COLUMBIA GAS CO. - TCO POOL (APPALACHIA) (BASIS) - ICE FUTURES ENERGY DIV',
    'NATURAL GAS PIPELINE-MID-CONTINENT POOL PIN (BASIS) - ICE FUTURES ENERGY DIV',
    'NORTHERN NATURAL GAS - DEMARCATION POOL (BASIS) - ICE FUTURES ENERGY DIV',
    'PANHANDLE EASTERN- POOL GAS (INDEX) - ICE FUTURES ENERGY DIV',
    'RBOB CALENDAR - NEW YORK MERCANTILE EXCHANGE',
    'ONEOK GAS TRANSPORTATION BASIS - ICE FUTURES ENERGY DIV',
    'NATURAL GAS INDEX: EP SAN JUAN - ICE FUTURES ENERGY DIV',
    'RBOB GASOLINE 1ST LINE - ICE FUTURES ENERGY DIV',
    'NATURAL GAS HENRY LD1 FIXED - ICE FUTURES ENERGY DIV',
    'NATURAL GAS PENULTIMATE ICE - ICE FUTURES ENERGY DIV',
    'GASOLINE BLENDSTOCK (RBOB) - NEW YORK MERCANTILE EXCHANGE',
    'USD Malaysian Crude Palm Oil C - CHICAGO MERCANTILE EXCHANGE',
    'CRUDE OIL CAL SPREAD OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'CRUDE DIFF-WCS HOUSTON/WTI 1ST - ICE FUTURES ENERGY DIV',
    'CRUDE OIL CAL SPREAD OPT FIN - NEW YORK MERCANTILE EXCHANGE',
    'CRUDE DIFF-TMX WCS 1A INDEX - ICE FUTURES ENERGY DIV',
    'CRUDE DIFF-TMX SW 1A INDEX - ICE FUTURES ENERGY DIV',
    'EUR STYLE CRUDE OIL OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'USD MALAYSIAN CRUDE PALM OIL - CHICAGO MERCANTILE EXCHANGE',
    'CRUDE OIL AVG PRICE OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'USD Malaysian Crude Palm Oil C - CHICAGO MERCANTILE EXCHANGE',
    'CRUDE OIL CAL SPREAD OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'FUEL OIL-3% USGC/3.5% FOB RDAM - ICE FUTURES ENERGY DIV',
    'MT BELV NAT GASOLINE OPIS - NEW YORK MERCANTILE EXCHANGE',
     'E-MINI S&P MATERIALS INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P FINANCIAL INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P UTILITIES INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P 400 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P ENERGY INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P CONSU STAPLES INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P TECHNOLOGY INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P HEALTH CARE INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P INDUSTRIAL INDEX - CHICAGO MERCANTILE EXCHANGE', 
     'EMINI RUSSELL 1000 VALUE INDEX - CHICAGO MERCANTILE EXCHANGE',
     'ADJUSTED INT RATE S&P 500 TOTL - CHICAGO MERCANTILE EXCHANGE',
     'S&P 500 ANNUAL DIVIDEND INDEX - CHICAGO MERCANTILE EXCHANGE',
     'S&P 500 QUARTERLY DIVIDEND IND - CHICAGO MERCANTILE EXCHANGE', 
     'DOW JONES U.S. REAL ESTATE IDX - CHICAGO BOARD OF TRADE',
     'DOW JONES INDUSTRIAL AVG- x $5 - CHICAGO BOARD OF TRADE',  
     'NIKKEI STOCK AVERAGE YEN DENOM - CHICAGO MERCANTILE EXCHANGE',
     'COLUMBIA GULF TRANSMISSION CO. -  MAINLINE POOL - ICE FUTURES ENERGY DIV',
     'PACIFIC GAS TRANSMISSION - MALIN (BASIS) - ICE FUTURES ENERGY DIV',
    'COPPER-GRADE #1 - COMMODITY EXCHANGE INC.',
    'LITHIUM HYDROXIDE - COMMODITY EXCHANGE INC.',
    'WTI FINANCIAL CRUDE OIL - NEW YORK MERCANTILE EXCHANGE',
    'WTI FINANCIAL CRUDE OIL - NEW YORK MERCANTILE EXCHANGE',
    'WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV',
    'BRENT CRUDE OIL LAST DAY - NEW YORK MERCANTILE EXCHANGE',
    'MARINE FUEL OIL 0.5% FOB USGC - ICE FUTURES ENERGY DIV',
    'GULF JET NY HEAT OIL SPR - NEW YORK MERCANTILE EXCHANGE',
 'CRUDE OIL CAL SPREAD OPT FIN - NEW YORK MERCANTILE EXCHANGE',
 'EUR STYLE CRUDE OIL OPTIONS - NEW YORK MERCANTILE EXCHANGE',
 'WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV',
 'USD MALAYSIAN CRUDE PALM OIL - CHICAGO MERCANTILE EXCHANGE',
 'BRENT CRUDE OIL LAST DAY - NEW YORK MERCANTILE EXCHANGE',
    'FUEL OIL-3% USGC/3.5% FOB RDAM BARGES - ICE FUTURES ENERGY DIV',
    'EUR STYLE NATURAL GAS OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'NAT GAS LD1 for GDD -TEXOK - ICE FUTURES ENERGY DIV',
 'NAT GAS ICE LD1 - ICE FUTURES ENERGY DIV',
 'NATURAL GAS CAL SPREAD OPT FIN - NEW YORK MERCANTILE EXCHANGE',
 'RBOB GASOLINE 1ST LINE - ICE FUTURES ENERGY DIV',
 'GASOLINE BLENDSTOCK (RBOB) - NEW YORK MERCANTILE EXCHANGE',
 'GASOLINE CRK-RBOB/BRENT 1st - ICE FUTURES ENERGY DIV',
 'BRENT LAST DAY - NEW YORK MERCANTILE EXCHANGE',
 'BRENT CRUDE OIL LAST DAY - NEW YORK MERCANTILE EXCHANGE',
    'MICRO GOLD - COMMODITY EXCHANGE INC.',
    'WTI 1st Line-Brent 1st Line - ICE FUTURES ENERGY DIV',
    'MICRO E-MINI RUSSELL 2000 INDX - CHICAGO MERCANTILE EXCHANGE',
    'E-MINI S&P MATERIALS INDEX - CHICAGO MERCANTILE EXCHANGE',
    'MICRO E-MINI DJIA (x$0.5) - CHICAGO BOARD OF TRADE',
    'E-MINI S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE',
    'S&P 500 Consolidated - CHICAGO MERCANTILE EXCHANGE',
    'S&P 500 TOTAL RETURN INDEX - CHICAGO MERCANTILE EXCHANGE',
    'NASDAQ-100 Consolidated - CHICAGO MERCANTILE EXCHANGE',
    'NFX CS5TC CAPESIZE 5T/C AVG - NASDAQ FUTURES',
    'NFX PM4TC PANAMAX 4T/C AVG - NASDAQ FUTURES',
    'NORTHWEST PIPELINE - CANADIAN BORDER (BASIS) - ICE FUTURES ENERGY DIV',
    'EURO FX/JAPANESE YEN XRATE - CHICAGO MERCANTILE EXCHANGE',
    'CRUDE DIFF-WCS CUSHING/WTI 1ST - ICE FUTURES ENERGY DIV',
    'DUTCH TTF NAT GAS CAL MONTH - NEW YORK MERCANTILE EXCHANGE',
    'TRANSCONTINENTAL GAS - ZONE 6 (NY) (BASIS) - ICE FUTURES ENERGY DIV',
    'GULF COAST UNL 87 GAS M2 PL RB - NEW YORK MERCANTILE EXCHANGE',
    'GULF COAST CBOB GAS A2 PL RBOB - NEW YORK MERCANTILE EXCHANGE',
    'NATURAL GAS INDEX: ALGONQUIN CITY GATES - ICE FUTURES ENERGY DIV',
    'HOT ROLLED COIL STEEL - NEW YORK MERCANTILE EXCHANGE',
    '3.5% FUEL OIL RDAM CRACK SPR - NEW YORK MERCANTILE EXCHANGE',
    'HENRY HUB PENULTIMATE NAT GAS - NEW YORK MERCANTILE EXCHANGE',
    'NAT GASLNE OPIS MT B NONTET FP - ICE FUTURES ENERGY DIV',
     'ULTRA U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE',
    'ULTRA U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE',
    'SOUTH AFRICAN RAND - CHICAGO MERCANTILE EXCHANGE',
    'Nano Bitcoin - LMX LABS LLC',
    'NANO ETHER - LMX LABS LLC',
    '2 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
    '#2 HEATING OIL- NY HARBOR-ULSD - NEW YORK MERCANTILE EXCHANGE',
    '#2 HEATING OIL- NY HARBOR-ULSD - NEW YORK MERCANTILE EXCHANGE',
    'CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE EXCHANGE',
    'WCS OIL NET ENERGY MONTHLY IND - NEW YORK MERCANTILE EXCHANGE',
    'NATURAL GAS - NEW YORK MERCANTILE EXCHANGE',
    'HHUB NAT GAS PENULT FINL-10000 - NASDAQ FUTURES',
    'MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'E-MINI RUSSELL 2000 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'MICRO E-MINI S&P 500 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'WTI  HOUSTON ARGUS/WTI TR MO - NEW YORK MERCANTILE EXCHANGE',
    'U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE',
    'MICRO E-MINI S&P 500 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE',
    'MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'NASDAQ-100 STOCK INDEX (MINI) - CHICAGO MERCANTILE EXCHANGE',
    'HOUSTON SHIP CHANNEL (INDEX) - ICE FUTURES ENERGY DIV',
    'BRITISH POUND STERLING - CHICAGO MERCANTILE EXCHANGE',
     'HHUB NAT GAS PENULT FINL-10000 - NASDAQ FUTURES',
     'NAT GAS ICE PEN - ICE FUTURES ENERGY DIV',
    'GULF COAST CBOB GAS A2 PL RBOB - NEW YORK MERCANTILE EXCHANGE',
    'GASOLINE CRK-RBOB/BRENT 1st - ICE FUTURES ENERGY DIV',
     'E-MINI S&P INDUSTRIAL INDEX - CHICAGO MERCANTILE EXCHANGE',
     'ERCOT Houston 345KV Hub RT 7x8 - ICE FUTURES ENERGY DIV',
    'ERCOT Houston 345KV RT OFF FIX - ICE FUTURES ENERGY DIV',
    'ERCOT HOUSTON 345KV RT PK FIX - ICE FUTURES ENERGY DIV',
     'CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE',
 'CRUDE OIL AVG PRICE OPTIONS - NEW YORK MERCANTILE EXCHANGE',
 'CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE',
  'GULF # 6 FUEL OIL CRACK - NEW YORK MERCANTILE EXCHANGE',
 '10 YEAR DELIVERABLE IR - CHICAGO BOARD OF TRADE',
 'TEXAS EASTERN- M3 ZONE (DELIVERED) (BASIS) - ICE FUTURES ENERGY DIV',
 'TEXAS EASTERN- M3 ZONE (DELIVERED) - ICE FUTURES ENERGY DIV',
 "WAHA HUB - WEST TEXAS DELIVERED/BUYER'S INDEX - ICE FUTURES ENERGY DIV",
 '5 YEAR DELIVERABLE IR - CHICAGO BOARD OF TRADE',
 'MICRO 10 YEAR YIELD - CHICAGO BOARD OF TRADE', 
 'MICRO E-MINI DJIA (x$0.5) - CHICAGO BOARD OF TRADE',
 'MICRO SING FOB MARINE FUEL .5% - NEW YORK MERCANTILE EXCHANGE',
 '10 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
 'MICRO E-MINI RUSSELL 2000 INDX - CHICAGO MERCANTILE EXCHANGE',
 'EMINI RUSSELL 1000 VALUE INDEX - CHICAGO MERCANTILE EXCHANGE',
 'E-MINI RUSSELL 2000 INDEX - CHICAGO MERCANTILE EXCHANGE',
 'MICRO E-MINI S&P 500 INDEX - CHICAGO MERCANTILE EXCHANGE',
 'CRUDE DIFF-WCS HOUSTON/WTI 1ST - ICE FUTURES ENERGY DIV',
 '5 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
 'DOW JONES INDUSTRIAL AVG- x $5 - CHICAGO BOARD OF TRADE',
 'ARGUS WTI HOUSTON/WTI TRADE MO - ICE FUTURES ENERGY DIV',
 'U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE',
 '10-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE',
 '5-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE',
 '2-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE',
 'UST BOND - CHICAGO BOARD OF TRADE',
 'ULTRA UST 10Y - CHICAGO BOARD OF TRADE',
 'MICRO E-MINI RUSSELL 2000 INDX - CHICAGO MERCANTILE EXCHANGE',
 'NASDAQ MINI - CHICAGO MERCANTILE EXCHANGE',
 'ULTRA UST BOND - CHICAGO BOARD OF TRADE',
 'ADJUSTED INT RATE S&P 500 TOTL - CHICAGO MERCANTILE EXCHANGE',
     'ADJUSTED INT RATE S&P 500 TOTL - CHICAGO MERCANTILE EXCHANGE',
    'GASOLINE BLENDSTOCK (RBOB) - NEW YORK MERCANTILE EXCHANGE',
    'GASOLINE CRK-RBOB/BRENT 1st - ICE FUTURES ENERGY DIV',
    'GULF COAST CBOB GAS A2 PL RBOB - NEW YORK MERCANTILE EXCHANGE',
    'RBOB CALENDAR - NEW YORK MERCANTILE EXCHANGE',
    'RBOB GASOLINE 1ST LINE - ICE FUTURES ENERGY DIV',
    'FUEL OIL USGC HSFO PLATTS BALM - ICE FUTURES ENERGY DIV',
    'CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE',
    'CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE EXCHANGE',
    'E-MINI S&P REAL ESTATE INDEX - CHICAGO MERCANTILE EXCHANGE',
    'E-MINI S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE',
    'NORTH EURO HOT-ROLL COIL STEEL - COMMODITY EXCHANGE INC.',
    'EUROPEAN PROPANE CIF ARA - NEW YORK MERCANTILE EXCHANGE',
    'ALUMINIUM EURO PREM DUTYUNPAID - COMMODITY EXCHANGE INC.',
    'E-MINI S&P COMMUNICATION INDEX - CHICAGO MERCANTILE EXCHANGE',
    'ULTRA UST BOND - CHICAGO BOARD OF TRADE',
    'RANDOM LENGTH LUMBER - CHICAGO MERCANTILE EXCHANGE',
    'SOFR-3M - CHICAGO MERCANTILE EXCHANGE',
    'SOFR-1M - CHICAGO MERCANTILE EXCHANGE',
    '1-MONTH SOFR - CHICAGO MERCANTILE EXCHANGE',
    '3-MONTH SOFR - CHICAGO MERCANTILE EXCHANGE',
    'NANO BITCOIN PERP STYLE - COINBASE DERIVATIVES, LLC',
    'NANO ETHER - COINBASE DERIVATIVES, LLC',
    'NANO ETHER PERP STYLE - COINBASE DERIVATIVES, LLC',
'NORTH EURO HOT-ROLL COIL STEEL - COMMODITY EXCHANGE INC.',
'Nano Bitcoin - COINBASE DERIVATIVES, LLC',
'RUSSELL 2000 ANNUAL DIVIDEND - CHICAGO MERCANTILE EXCHANGE',
'ULTRA US T BOND - CHICAGO BOARD OF TRADE',
'ULTRA UST BOND - CHICAGO BOARD OF TRADE',
'WHEAT-HRSpring - MIAX FUTURES EXCHANGE',
'WTI  HOUSTON ARGUS/WTI BALMO - NEW YORK MERCANTILE EXCHANGE',
'3 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
'EURO FX - CHICAGO MERCANTILE EXCHANGE',
'EURO FX/JAPANESE YEN XRATE - CHICAGO MERCANTILE EXCHANGE',
'MICRO SOL - CHICAGO MERCANTILE EXCHANGE',
'MICRO XRP - CHICAGO MERCANTILE EXCHANGE',
'BITCOIN CASH PERP STYLE - COINBASE DERIVATIVES, LLC',
'NAT GAS TETCO-WLA INDEX - ICE FUTURES ENERGY DIV',
'GRP 3 SOC GAS VS RBOB SPR - NEW YORK MERCANTILE EXCHANGE',
'GOLD -1 TROY OUNCE - COINBASE DERIVATIVES'





    
    
]


In [58]:
for item in remove_items:
    if item in unique_markets:
        unique_markets.remove(item)

In [59]:
sorted(list(dict.fromkeys(unique_markets))) #this is the market list for making graphs.

['7 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
 'AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE',
 'BITCOIN - CHICAGO MERCANTILE EXCHANGE',
 'BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE',
 'BRITISH POUND - CHICAGO MERCANTILE EXCHANGE',
 'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE',
 'COCOA - ICE FUTURES U.S.',
 'COFFEE C - ICE FUTURES U.S.',
 'COPPER- #1 - COMMODITY EXCHANGE INC.',
 'CORN - CHICAGO BOARD OF TRADE',
 'COTTON NO. 2 - ICE FUTURES U.S.',
 'E-MINI NATURAL GAS - NEW YORK MERCANTILE EXCHANGE',
 'E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE',
 'E-MINI S&P CONSUMER DISC INDEX - CHICAGO MERCANTILE EXCHANGE',
 'EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE',
 'ERCOT.HB_HOUSTON_month_on_rtp - NODAL EXCHANGE',
 'ETHER CASH SETTLED - CHICAGO MERCANTILE EXCHANGE',
 'EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE',
 'FEEDER CATTLE - CHICAGO MERCANTILE EXCHANGE',
 'GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE',
 'GOLD - COMMODITY EXCHANGE INC.',
 'GOLD -1 

In [60]:
#remove duplicates
unique_markets = list(dict.fromkeys(unique_markets))

# Markets that IB has no usable data for (no listing, wrong product type, no
# subscription, etc.). These are removed up-front so every downstream cell
# (mapping, fetcher, diagnostic, signals) ignores them silently.
IGNORE_MARKETS = [
    'GOLD -1 TROY OUNCE - COINBASE DERIVATIVES, LLC',
    'E-MINI S&P CONSUMER DISC INDEX - CHICAGO MERCANTILE EXCHANGE',
    '7 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
    'ETHER CASH SETTLED - CHICAGO MERCANTILE EXCHANGE',
    # Full-size Silver (SI) — heavy IB load and dual-class noise; MICRO SILVER (QI)
    # is the tradable size.
    'SILVER - COMMODITY EXCHANGE INC.',
]
unique_markets = [m for m in unique_markets if m not in IGNORE_MARKETS]

### 4. Create Open Interest Index Value for a Commodity 

In [61]:
#create new DF for this part of the analysis
df2 = df.sort_values(['Market', 'Datetime'], ascending = [True, True])

In [62]:
#Group markerss and add Open Interest Index Column

group = df2.groupby("Market")["OI"]
df2["OI_Index"] = group.transform(lambda x: (((x - min(x)) / (max(x) - min(x))) * 100))

## Forumla for indexing:
#zi = (xi – min(x)) / (max(x) – min(x)) * 100 = (12 – 12) / (68 – 12) * 100 = 0


### 5. See if a market has a unique value based on your keyword or search term


In [63]:
def number_of_markets (keyword_or_phrase):
    
    filtered = df[df["Market"].str.contains(keyword_or_phrase, case=False)]
    filtered = filtered["Market"].unique().tolist()
    
    return filtered

In [64]:
number_of_markets("cocoa")

['COCOA - ICE FUTURES U.S.']

### Retail OI indexing

In [65]:
df2.columns.to_list()

['Market',
 'Datetime',
 'Date',
 'CFTC Contract Market Code',
 'CFTC Market Code in Initials',
 'CFTC Region Code',
 'CFTC Commodity Code',
 'OI',
 'Noncommercial Positions-Long (All)',
 'Noncommercial Positions-Short (All)',
 'Noncommercial Positions-Spreading (All)',
 'Commercial Positions-Long (All)',
 'Commercial Positions-Short (All)',
 ' Total Reportable Positions-Long (All)',
 'Total Reportable Positions-Short (All)',
 'Nonreportable Positions-Long (All)',
 'Nonreportable Positions-Short (All)',
 'Open Interest (Old)',
 'Noncommercial Positions-Long (Old)',
 'Noncommercial Positions-Short (Old)',
 'Noncommercial Positions-Spreading (Old)',
 'Commercial Positions-Long (Old)',
 'Commercial Positions-Short (Old)',
 'Total Reportable Positions-Long (Old)',
 'Total Reportable Positions-Short (Old)',
 'Nonreportable Positions-Long (Old)',
 'Nonreportable Positions-Short (Old)',
 'Open Interest (Other)',
 'Noncommercial Positions-Long (Other)',
 'Noncommercial Positions-Short (Other)'

In [66]:
cols = ['Market', 'Datetime' , 'Noncommercial Positions-Long (All)',
 'Noncommercial Positions-Short (All)',
 'Noncommercial Positions-Spreading (All)',
 'Commercial Positions-Long (All)',
 'Commercial Positions-Short (All)',
 ' Total Reportable Positions-Long (All)',
 'Total Reportable Positions-Short (All)',
 'Nonreportable Positions-Long (All)',
 'Nonreportable Positions-Short (All)',
        'OI_Index', "OI"]

In [67]:
df3 = df2[cols].copy()

In [68]:
row_test = df3.iloc[0, 1:10] #list of column headers and 1st row of data

In [69]:
df3["Net Retail Position"] = df3["Nonreportable Positions-Long (All)"] - df3["Nonreportable Positions-Short (All)"]

In [70]:
group2 = df3.groupby("Market")["Net Retail Position"]
df3["Retail_Index"] = group2.transform(lambda x: (((x - min(x)) / (max(x) - min(x))) * 100))

In [71]:
df3["Net Commercial Position"] = df3["Commercial Positions-Long (All)"] - df3["Commercial Positions-Short (All)"]


In [72]:
group3 = df3.groupby("Market")["Net Commercial Position"]
df3["Commercial_Index"] = group3.transform(lambda x: (((x - min(x)) / (max(x) - min(x))) * 100))

In [73]:
df3["Net Traders Position"] = df3["Noncommercial Positions-Long (All)"] - df3["Noncommercial Positions-Short (All)"]


In [74]:
group4 = df3.groupby("Market")["Net Traders Position"]
df3["Traders_Index"] = group4.transform(lambda x: (((x - min(x)) / (max(x) - min(x))) * 100))

In [75]:
df3.tail()

,Market,Datetime,Noncommercial Positions-Long (All),Noncommercial Positions-Short (All),Noncommercial Positions-Spreading (All),Commercial Positions-Long (All),Commercial Positions-Short (All),Total Reportable Positions-Long (All),Total Reportable Positions-Short (All),Nonreportable Positions-Long (All),Nonreportable Positions-Short (All),OI_Index,OI,Net Retail Position,Retail_Index,Net Commercial Position,Commercial_Index,Net Traders Position,Traders_Index
70118,"ZCASH PERP STYLE - COINBASE DERIVATIVES, LLC",2026-04-21,2549,2902,0,0,0,2549,2902,619,266,80.464886,3168,353,100.000000,0,NaN,-353,0.000000
70117,"ZCASH PERP STYLE - COINBASE DERIVATIVES, LLC",2026-04-28,2040,2342,0,0,0,2040,2342,588,286,53.758655,2628,302,89.055794,0,NaN,-302,10.944206
70116,"ZCASH PERP STYLE - COINBASE DERIVATIVES, LLC",2026-05-05,2390,2277,0,0,0,2390,2277,591,704,71.216617,2981,-113,0.000000,0,NaN,113,100.000000
70115,"ZCASH PERP STYLE - COINBASE DERIVATIVES, LLC",2026-05-12,1931,2184,0,0,0,1931,2184,778,525,57.764590,2709,253,78.540773,0,NaN,-253,21.459227
70114,"ZCASH PERP STYLE - COINBASE DERIVATIVES, LLC",2026-05-19,2683,2974,0,0,0,2683,2974,880,589,100.000000,3563,291,86.695279,0,NaN,-291,13.304721


### Summarise Latest Week's Data in a Table

In [76]:
OI_condition = df3[(df3['OI_Index'] >= 80) | (df3['OI_Index']<=20)]
Retail_condition = df3[(df3['Retail_Index'] >= 80) | (df3['Retail_Index']<=20)]
Commercial_condition = df3[(df3['Commercial_Index'] >= 80) | (df3['Commercial_Index']<=20)]
Date_coundition =df3["Datetime"].max() 

In [77]:
# Get the most recent date
most_recent_date = df3['Datetime'].max()

# Filter the DataFrame for the most recent date and conditions

summary_table = df3.loc[(df3['Datetime'] == most_recent_date) & 
                        (((df3['OI_Index'] >= 80) | (df3['OI_Index'] <= 20)) |
                         ((df3['Retail_Index'] >= 80) | (df3['Retail_Index'] <= 20)) |
                         ((df3['Commercial_Index'] >= 80) | (df3['Commercial_Index'] <= 20))),
                        ['Market', 'OI_Index', 'Retail_Index', 'Commercial_Index']]


In [78]:
summary_table_filtered = summary_table[summary_table['Market'].isin(unique_markets)]
summary_table_filtered['Market'] = summary_table_filtered['Market'].str.split(' -').str[0].str.strip()


/var/folders/km/x_cb0wn5249ddhfzzt066kg40000gn/T/ipykernel_82930/3186465317.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  summary_table_filtered['Market'] = summary_table_filtered['Market'].str.split(' -').str[0].str.strip()


### IB Gateway Contract Mapping

In [79]:
# COT Market name → (IB_Symbol, IB_Exchange, IB_Currency)
ib_mapping = {
    # Metals
    'GOLD - COMMODITY EXCHANGE INC.':        ('GC',  'COMEX', 'USD'),
    # SILVER full-size omitted — IGNORE_MARKETS / use MICRO SILVER (QI).
    'PLATINUM - NEW YORK MERCANTILE EXCHANGE': ('PL', 'NYMEX', 'USD'),
    'PALLADIUM - NEW YORK MERCANTILE EXCHANGE': ('PA', 'NYMEX', 'USD'),
    'COPPER- #1 - COMMODITY EXCHANGE INC.':  ('HG',  'COMEX', 'USD'),
    'MICRO GOLD - COMMODITY EXCHANGE INC.':  ('MGC', 'COMEX', 'USD'),
    'MICRO SILVER - COMMODITY EXCHANGE INC.': ('QI', 'COMEX', 'USD'),
    'MICRO COPPER - COMMODITY EXCHANGE INC.': ('MHG', 'COMEX', 'USD'),

    # Energies
    'WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE':     ('CL', 'NYMEX', 'USD'),
    'GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE':    ('RB', 'NYMEX', 'USD'),
    'NAT GAS NYME - NEW YORK MERCANTILE EXCHANGE':     ('NG', 'NYMEX', 'USD'),
    'E-MINI NATURAL GAS - NEW YORK MERCANTILE EXCHANGE': ('QG', 'NYMEX', 'USD'),

    # Grains
    'CORN - CHICAGO BOARD OF TRADE':       ('ZC', 'CBOT', 'USD'),
    'OATS - CHICAGO BOARD OF TRADE':       ('ZO', 'CBOT', 'USD'),
    'WHEAT-SRW - CHICAGO BOARD OF TRADE':  ('ZW', 'CBOT', 'USD'),
    'SOYBEANS - CHICAGO BOARD OF TRADE':   ('ZS', 'CBOT', 'USD'),
    'SOYBEAN MEAL - CHICAGO BOARD OF TRADE': ('ZM', 'CBOT', 'USD'),
    'SOYBEAN OIL - CHICAGO BOARD OF TRADE':  ('ZL', 'CBOT', 'USD'),
    'MINI SOYBEANS - CHICAGO BOARD OF TRADE': ('ZS', 'CBOT', 'USD'),

    # Softs
    'COCOA - ICE FUTURES U.S.':      ('CC', 'NYBOT', 'USD'),
    'COFFEE C - ICE FUTURES U.S.':   ('KC', 'NYBOT', 'USD'),
    'COTTON NO. 2 - ICE FUTURES U.S.': ('CT', 'NYBOT', 'USD'),
    'SUGAR NO. 11 - ICE FUTURES U.S.': ('SB', 'NYBOT', 'USD'),

    # Livestock
    'FEEDER CATTLE - CHICAGO MERCANTILE EXCHANGE': ('GF', 'CME', 'USD'),
    'LEAN HOGS - CHICAGO MERCANTILE EXCHANGE':     ('HE', 'CME', 'USD'),
    'LIVE CATTLE - CHICAGO MERCANTILE EXCHANGE':   ('LE', 'CME', 'USD'),

    # Indices
    'E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE':  ('ES', 'CME', 'USD'),
    'RUSSELL E-MINI - CHICAGO MERCANTILE EXCHANGE':   ('RTY', 'CME', 'USD'),
    'MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE': ('MNQ', 'CME', 'USD'),
    'NIKKEI STOCK AVERAGE - CHICAGO MERCANTILE EXCHANGE': ('NKD', 'CME', 'USD'),
    'EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE': ('RTY', 'CME', 'USD'),
    'VIX FUTURES - CBOE FUTURES EXCHANGE': ('VIX', 'CFE', 'USD'),

    # Currencies — ContFuture needs (ISO code, exchange, ccy, tradingClass=exchange symbol)
    'AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE':  ('AUD', 'CME', 'USD', '6A'),
    'BRITISH POUND - CHICAGO MERCANTILE EXCHANGE':      ('GBP', 'CME', 'USD', '6B'),
    'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE':    ('CAD', 'CME', 'USD', '6C'),
    'EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE': ('EUR', 'CME', 'USD', '6E'),
    'JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE':       ('JPY', 'CME', 'USD', '6J'),
    'MEXICAN PESO - CHICAGO MERCANTILE EXCHANGE':       ('MXP', 'CME', 'USD', '6M'),
    'NEW ZEALAND DOLLAR - CHICAGO MERCANTILE EXCHANGE': ('NZD', 'CME', 'USD', '6N'),
    'SWISS FRANC - CHICAGO MERCANTILE EXCHANGE':        ('CHF', 'CME', 'USD', '6S'),
    'SO AFRICAN RAND - CHICAGO MERCANTILE EXCHANGE':    ('ZAR', 'CME', 'USD', '6Z'),
    'BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE':     ('BRL', 'CME', 'USD', '6L'),

    # Crypto (CME futures, not spot). ETHER CASH SETTLED is intentionally
    # excluded — the underlying 'ETH' resolves to a cash index, not a tradable
    # future. MICRO ETHER (MET) is the contract actually traded.
    'BITCOIN - CHICAGO MERCANTILE EXCHANGE':        ('BRR', 'CME', 'USD'),
    'MICRO BITCOIN - CHICAGO MERCANTILE EXCHANGE':  ('MBT', 'CME', 'USD'),
    'MICRO ETHER - CHICAGO MERCANTILE EXCHANGE':    ('MET', 'CME', 'USD'),

    # Financials
    'UST 2Y NOTE - CHICAGO BOARD OF TRADE':  ('ZT', 'CBOT', 'USD'),
    'UST 5Y NOTE - CHICAGO BOARD OF TRADE':  ('ZF', 'CBOT', 'USD'),
    'UST 10Y NOTE - CHICAGO BOARD OF TRADE': ('ZN', 'CBOT', 'USD'),
    'UST BOND - CHICAGO BOARD OF TRADE':     ('ZB', 'CBOT', 'USD'),
}

In [80]:
mapping_df = pd.DataFrame([
    {
        'Market': k,
        'IB_Symbol': v[3] if len(v) > 3 else v[0],
        'IB_Exchange': v[1],
        'IB_Currency': v[2],
        'IB_ContFutSymbol': v[0],
        'IB_TradingClass': v[3] if len(v) > 3 else '',
    }
    for k, v in ib_mapping.items()
])

In [81]:
market_categories = {
    'GOLD - COMMODITY EXCHANGE INC.': 'Metals',
    # SILVER full-size omitted with IGNORE_MARKETS (MICRO SILVER remains).
    'PLATINUM - NEW YORK MERCANTILE EXCHANGE': 'Metals',
    'PALLADIUM - NEW YORK MERCANTILE EXCHANGE': 'Metals',
    'COPPER- #1 - COMMODITY EXCHANGE INC.': 'Metals',
    'SOYBEAN OIL - CHICAGO BOARD OF TRADE': 'Softs',
    'SOYBEANS - CHICAGO BOARD OF TRADE': 'Softs',
    'SOYBEAN MEAL - CHICAGO BOARD OF TRADE': 'Softs',
    'WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE': 'Energies',
    'GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE': 'Energies',
    'NAT GAS NYME - NEW YORK MERCANTILE EXCHANGE': 'Energies',
    'CORN - CHICAGO BOARD OF TRADE': 'Grains',
    'OATS - CHICAGO BOARD OF TRADE': 'Grains',
    'WHEAT-SRW - CHICAGO BOARD OF TRADE': 'Grains',
    'FEEDER CATTLE - CHICAGO MERCANTILE EXCHANGE': 'Livestock',
    'LEAN HOGS - CHICAGO MERCANTILE EXCHANGE': 'Livestock',
    'LIVE CATTLE - CHICAGO MERCANTILE EXCHANGE': 'Livestock',
    'COCOA - ICE FUTURES U.S.': 'Softs',
    'COFFEE C - ICE FUTURES U.S.': 'Softs',
    'COTTON NO. 2 - ICE FUTURES U.S.': 'Softs',
    'SUGAR NO. 11 - ICE FUTURES U.S.': 'Softs',
    'RUSSELL E-MINI - CHICAGO MERCANTILE EXCHANGE': 'Indices',
    'E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE': 'Indices',
    'MICRO ETHER - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'MICRO BITCOIN - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'MICRO GOLD - COMMODITY EXCHANGE INC.': 'Metals',
    'MICRO SILVER - COMMODITY EXCHANGE INC.': 'Metals',
    'MICRO COPPER - COMMODITY EXCHANGE INC.': 'Metals',
    'MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE': 'Indices',
    'NIKKEI STOCK AVERAGE - CHICAGO MERCANTILE EXCHANGE': 'Indices',
    'AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'UST 5Y NOTE - CHICAGO BOARD OF TRADE': 'Financials',
    'UST 2Y NOTE - CHICAGO BOARD OF TRADE': 'Financials',
    'UST 10Y NOTE - CHICAGO BOARD OF TRADE': 'Financials',
    'UST BOND - CHICAGO BOARD OF TRADE': 'Financials',
    'MEXICAN PESO - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'SWISS FRANC - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'BRITISH POUND - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'NEW ZEALAND DOLLAR - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'SO AFRICAN RAND - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'BITCOIN - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'VIX FUTURES - CBOE FUTURES EXCHANGE': 'Indices',
     'MINI SOYBEANS - CHICAGO BOARD OF TRADE': 'Softs',
 'EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE': 'Indices',
 'E-MINI NATURAL GAS - NEW YORK MERCANTILE EXCHANGE': 'Energies'
}


In [82]:
#map the categories to the market in the mapping df
mapping_df["group"] = mapping_df["Market"].map(market_categories)

In [83]:
df4 = df3.copy()

In [84]:
df4 = pd.merge(df4, mapping_df, on="Market", how='left')

In [85]:
df4.rename(columns = {"Datetime" : "Date"}, inplace = True)

In [86]:
df4['group'].fillna('Financials', inplace=True)

/var/folders/km/x_cb0wn5249ddhfzzt066kg40000gn/T/ipykernel_82930/638600763.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df4['group'].fillna('Financials', inplace=True)


## Create RSI Function


In [87]:
def rsi(data, periods=10):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=periods).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=periods).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

## Download Price Data from IB Gateway

In [88]:
# =============================================================================
# Back-adjusted continuous-contract builder (Panama / difference method)
# =============================================================================
#
# Roll rule: each contract is rolled `roll_buffer_business_days` business days
#     before its lastTradeDateOrContractMonth. This single rule is applied
#     uniformly to physically-delivered and cash-settled contracts. It mirrors
#     the trader's actual exit discipline (flat 5 BD before notice / expiry,
#     re-enter only on a fresh signal).
#
# Adjustment: walking newest -> oldest, accumulate
#     gap = close(c_next, t_roll) - close(c, t_roll)
#     and add the running cumulative offset to OHLC of every bar belonging to
#     `c` (or any earlier contract). Volume is NOT adjusted because volume is
#     a per-contract quantity. The newest contract has offset = 0, so recent
#     prices on the adjusted series equal the actual current front-month price.
#
# This is the same series in backtest and live: there is no front-month vs
# stitched divergence to manage.

def _bday_offset(d, offset_days):
    """Return d shifted by offset_days business days (offset can be negative)."""
    if offset_days == 0:
        return d
    sign = 1 if offset_days > 0 else -1
    n = abs(offset_days)
    cur = d
    while n > 0:
        cur = cur + timedelta(days=sign)
        if cur.weekday() < 5:
            n -= 1
    return cur


def _bar_date(b):
    """Return the date portion of a bar's timestamp (works for daily and intraday)."""
    bd = b.date
    if hasattr(bd, 'date'):
        return bd.date()
    return bd


def _last_close_on(bars, target_date):
    """Last bar close on `target_date`, or None if no bar that day."""
    on_day = [b for b in bars if _bar_date(b) == target_date]
    if not on_day:
        return None
    return on_day[-1].close


def find_liquid_front_contract(
    ib, *, future_symbol, exchange, currency, trading_class='',
    lookback_days=10, forward_window_days=180,
    pacing_seconds=2.5,
):
    """Return (contract, expiry_date) for the listing with highest summed daily
    TRADES volume over the last `lookback_days` sessions, among contracts whose
    expiry falls in [today, today + forward_window_days]. Tie-break: nearest
    expiry. Returns (None, None) if lookup fails.
    """
    template = Future(symbol=future_symbol, exchange=exchange, currency=currency)
    if trading_class:
        template.tradingClass = trading_class
    template.includeExpired = False
    try:
        details = ib.reqContractDetails(template)
    except Exception:
        return None, None
    if not details:
        return None, None

    time.sleep(2.0)  # gap after reqContractDetails — fewer HMDS 162 cancels

    today = date.today()
    horizon = today + timedelta(days=forward_window_days)
    candidates = []
    for d in details:
        c = d.contract
        s = c.lastTradeDateOrContractMonth or ''
        try:
            if len(s) >= 8:
                exp = date(int(s[:4]), int(s[4:6]), int(s[6:8]))
            elif len(s) == 6:
                exp = date(int(s[:4]), int(s[4:6]), 28)
            else:
                continue
        except Exception:
            continue
        if today <= exp <= horizon:
            candidates.append((exp, c))

    if not candidates:
        return None, None

    duration = f'{max(lookback_days + 5, 15)} D'
    pace = max(float(pacing_seconds), 2.0)
    _sleep = getattr(ib, 'sleep', None) or time.sleep
    scored = []
    for exp, c in candidates:
        _sleep(pace)  # before each HMDS pull — avoids cancelling prior req
        blist = []
        for attempt in range(3):
            try:
                bars = ib.reqHistoricalData(
                    c, endDateTime='', durationStr=duration,
                    barSizeSetting='1 day', whatToShow='TRADES',
                    useRTH=False, formatDate=1, timeout=30,
                )
            except Exception:
                bars = []
            blist = list(bars or [])
            if blist:
                break
            _sleep(3.0 + attempt * 2.0)
        _sleep(pace)
        total = sum((b.volume or 0) for b in blist[-lookback_days:])
        scored.append((total, exp, c))

    scored.sort(key=lambda x: (-x[0], x[1]))
    _, exp, c = scored[0]
    return c, exp


def build_backadjusted_continuous(
    ib, *,
    future_symbol, exchange, currency, trading_class='',
    start_date, bar_size='1 day',
    what_to_show='TRADES', use_rth=False,
    roll_buffer_business_days=5,
    pacing_seconds=3.0,
    end_dt=None,
):
    """
    Build a Panama back-adjusted continuous price series from individual
    expiring contracts.

    Returns
    -------
    bars : list[dict]
        Each dict: {date, open, high, low, close, volume, contract_local_symbol}.
        Bars are sorted by date and de-duplicated.
    roll_log : list[dict]
        One entry per pairwise roll: {from, to, nominal_roll, actual_roll, gap}.
    status : str
        Human-readable status. 'ok' on success, otherwise an error reason.
    """
    template = Future(symbol=future_symbol, exchange=exchange, currency=currency)
    if trading_class:
        template.tradingClass = trading_class
    template.includeExpired = True

    try:
        details = ib.reqContractDetails(template)
    except Exception as e:
        return [], [], f'reqContractDetails error: {e}'
    if not details:
        return [], [], 'no contracts found'

    time.sleep(1.5)  # gap after reqContractDetails before historical barrage

    # ---- 1. Enumerate contracts whose lastTradeDate is in our window --------
    # Upper bound = liquid front expiry (+2d): never pull deep deferreds that
    # break roll-date logic. Expired listings still satisfy exp <= cap.
    liquid_front, front_exp = find_liquid_front_contract(
        ib, future_symbol=future_symbol, exchange=exchange,
        currency=currency, trading_class=trading_class,
        pacing_seconds=max(float(pacing_seconds), 2.5),
    )
    if liquid_front is None:
        return [], [], 'no liquid front contract found'
    contract_window_end = front_exp + timedelta(days=2)
    contracts = []
    for d in details:
        c = d.contract
        s = c.lastTradeDateOrContractMonth or ''
        try:
            if len(s) >= 8:
                exp = date(int(s[:4]), int(s[4:6]), int(s[6:8]))
            elif len(s) == 6:
                exp = date(int(s[:4]), int(s[4:6]), 28)
            else:
                continue
        except Exception:
            continue
        # For intraday only: skip already-expired contracts. IB's HMDS times out
        # on per-contract intraday requests for expired contracts (especially on
        # weekends), and ORB strategies don't need cross-roll continuity intraday.
        if bar_size != '1 day' and exp < date.today():
            continue
        if start_date <= exp <= contract_window_end:
            contracts.append((exp, c))
    if not contracts:
        return [], [], 'no contracts in window'

    contracts.sort(key=lambda x: x[0])
    seen = set()
    deduped = []
    for exp, c in contracts:
        key = c.localSymbol or (c.symbol, c.lastTradeDateOrContractMonth)
        if key in seen:
            continue
        seen.add(key)
        deduped.append((exp, c))
    contracts = deduped

    # ---- 2. Fetch bars per contract -----------------------------------------
    if bar_size == '1 day':
        duration = '1 Y'
    elif bar_size in ('30 mins', '1 hour'):
        duration = '30 D'   # IB caps intraday requests at ~1 month per call
    else:
        duration = '1 Y'

    today = date.today()
    contract_bars = {}  # local_symbol -> (contract, exp, bars)
    for exp, c in contracts:
        cap_end = end_dt if end_dt is not None else \
            datetime.combine(min(exp + timedelta(days=2), today), datetime.min.time())
        try:
            bars = ib.reqHistoricalData(
                c, endDateTime=cap_end, durationStr=duration,
                barSizeSetting=bar_size, whatToShow=what_to_show,
                useRTH=use_rth, formatDate=1,
                timeout=15,
            )
        except Exception:
            bars = []
        contract_bars[c.localSymbol] = (c, exp, list(bars or []))
        time.sleep(pacing_seconds)

    sorted_locals = sorted(
        [ls for ls, (_, _, bars) in contract_bars.items() if bars],
        key=lambda ls: contract_bars[ls][1],
    )
    if not sorted_locals:
        return [], [], 'no bars returned for any contract'

    # ---- 3. Compute roll dates and Panama offsets (newest -> oldest) --------
    offset_per_local = {sorted_locals[-1]: 0.0}
    roll_log = []

    for i in range(len(sorted_locals) - 2, -1, -1):
        cur_local = sorted_locals[i]
        nxt_local = sorted_locals[i + 1]
        _, cur_exp, cur_bars = contract_bars[cur_local]
        _, _, nxt_bars = contract_bars[nxt_local]

        nominal = _bday_offset(cur_exp, -roll_buffer_business_days)
        if nominal > date.today():
            offset_per_local[cur_local] = offset_per_local[nxt_local]
            roll_log.append({
                'from': cur_local, 'to': nxt_local,
                'nominal_roll': nominal, 'actual_roll': None,
                'gap': 0.0, 'note': 'nominal in future; skipped',
            })
            continue

        # Walk back day-by-day until both contracts have a close
        actual = None
        for k in range(15):
            cand = _bday_offset(nominal, -k)
            if (_last_close_on(cur_bars, cand) is not None
                    and _last_close_on(nxt_bars, cand) is not None):
                actual = cand
                break

        if actual is None:
            cur_dates = {_bar_date(b) for b in cur_bars}
            nxt_dates = {_bar_date(b) for b in nxt_bars}
            common = sorted(cur_dates & nxt_dates)
            if not common:
                offset_per_local[cur_local] = offset_per_local[nxt_local]
                roll_log.append({
                    'from': cur_local, 'to': nxt_local,
                    'nominal_roll': nominal, 'actual_roll': None,
                    'gap': 0.0, 'note': 'no overlap; offset propagated',
                })
                continue
            actual = max(d for d in common if d <= nominal) if any(d <= nominal for d in common) else common[-1]

        gap = _last_close_on(nxt_bars, actual) - _last_close_on(cur_bars, actual)
        offset_per_local[cur_local] = offset_per_local[nxt_local] + gap
        roll_log.append({
            'from': cur_local, 'to': nxt_local,
            'nominal_roll': nominal, 'actual_roll': actual, 'gap': gap,
        })

    # ---- 4. Stitch bars within each contract's active window ---------------
    locals_to_roll = {entry['from']: entry['actual_roll'] for entry in roll_log}

    out = []
    prev_roll = None
    for local in sorted_locals:
        _, _, bars = contract_bars[local]
        my_roll = locals_to_roll.get(local)  # None for newest
        offset = offset_per_local[local]
        for b in bars:
            d = _bar_date(b)
            if prev_roll is not None and d <= prev_roll:
                continue
            if my_roll is not None and d > my_roll:
                continue
            if d < start_date:
                continue
            out.append({
                'date': b.date,
                'open':  b.open  + offset,
                'high':  b.high  + offset,
                'low':   b.low   + offset,
                'close': b.close + offset,
                'volume': b.volume,
                'contract_local_symbol': local,
            })
        if my_roll is not None:
            prev_roll = my_roll

    out.sort(key=lambda x: x['date'])
    seen_keys = set()
    deduped_out = []
    for r in out:
        k = (r['date'], r['contract_local_symbol'])
        if k in seen_keys:
            continue
        seen_keys.add(k)
        deduped_out.append(r)

    return deduped_out, roll_log, 'ok'


# =============================================================================
# Helpers for cheap incremental updates (no full rebuild)
# =============================================================================
#
# The newest contract in a Panama-adjusted series has offset = 0 by
# construction. So as long as no roll has crossed since the last cache
# update, the current front month *is* the cached "newest contract", and
# we can fetch only the missing tail bars and append them with zero
# adjustment. A full rebuild is only needed when a roll has actually
# crossed -- the contract that was the front last run is no longer the
# front now. Under the 5 BD-before-lastTrade rule that happens about once
# per month per market.

def find_current_front_contract(
    ib, *, future_symbol, exchange, currency, trading_class='',
    roll_buffer_business_days=5,
):
    """Return (contract, expiry_date) of the current front month under our
    roll rule: earliest-expiry contract whose roll_date
    (= lastTrade - roll_buffer BDays) is strictly in the future.
    Returns (None, None) if lookup fails.
    """
    template = Future(symbol=future_symbol, exchange=exchange, currency=currency)
    if trading_class:
        template.tradingClass = trading_class
    template.includeExpired = False
    try:
        details = ib.reqContractDetails(template)
    except Exception:
        return None, None
    if not details:
        return None, None

    time.sleep(1.0)

    today = date.today()
    candidates = []
    for d in details:
        c = d.contract
        s = c.lastTradeDateOrContractMonth or ''
        try:
            if len(s) >= 8:
                exp = date(int(s[:4]), int(s[4:6]), int(s[6:8]))
            elif len(s) == 6:
                exp = date(int(s[:4]), int(s[4:6]), 28)
            else:
                continue
        except Exception:
            continue
        roll_dt = _bday_offset(exp, -roll_buffer_business_days)
        if roll_dt > today:
            candidates.append((exp, c))
    if not candidates:
        return None, None
    candidates.sort(key=lambda x: x[0])
    exp, c = candidates[0]
    return c, exp


def fetch_recent_bars(
    ib, contract, *, since_date, bar_size='1 day',
    what_to_show='TRADES', use_rth=False,
):
    """Fetch bars from `contract` strictly after `since_date`. Returns raw IB
    bar objects (offset = 0; caller must guarantee `contract` is the current
    front month).
    """
    today = date.today()
    gap_days = max((today - since_date).days, 1)
    if bar_size == '1 day':
        if gap_days <= 7:
            duration = '10 D'
        elif gap_days <= 30:
            duration = '1 M'
        elif gap_days <= 90:
            duration = '3 M'
        elif gap_days <= 180:
            duration = '6 M'
        else:
            duration = '1 Y'
    elif bar_size == '30 mins':
        duration = f'{min(max(gap_days + 2, 5), 60)} D'
    elif bar_size == '1 hour':
        duration = f'{min(max(gap_days + 2, 5), 90)} D'
    else:
        duration = '1 M'
    try:
        bars = ib.reqHistoricalData(
            contract, endDateTime='', durationStr=duration,
            barSizeSetting=bar_size, whatToShow=what_to_show,
            useRTH=use_rth, formatDate=1,
            timeout=15,
        )
    except Exception:
        bars = []
    return [b for b in (bars or []) if _bar_date(b) > since_date]


In [90]:
from pickle import FALSE
from ib_insync import *
from datetime import date, datetime, timedelta
import time, os, json as _json

IB_HOST = '127.0.0.1'
IB_PORT = 4001       # live trading
CLIENT_ID = 1
START_DATE = date(2022, 1, 1)
DAILY_CACHE_FILE = 'ib_daily_cache.json'
ROLL_STATE_FILE  = 'ib_daily_roll_state.json'

# Set True to wipe the cache and rebuild every market from scratch. Normally
# leave False: each market follows the cheapest viable path -- skip if cache
# is current through today, incremental tail-append if no roll has crossed
# since the last update, or full rebuild only if a roll has crossed (which
# happens about once a month per market).
# One-time recovery after deploying volume-based front logic: set True, run this
# cell once (with IB connected) to wipe ib_daily_cache.json + roll state, then
# set back to False.
FORCE_FULL_REBUILD = False

# Roll buffer in business days before lastTradeDateOrContractMonth. Mirrors
# the trader's discipline of being flat 5 BD before notice/expiry.
ROLL_BUFFER_BDAYS = 5

# ── Load existing cache (if any) ──────────────────────────────────────
if FORCE_FULL_REBUILD and os.path.exists(DAILY_CACHE_FILE):
    print("FORCE_FULL_REBUILD=True → wiping cache.")
    os.remove(DAILY_CACHE_FILE)
    if os.path.exists(ROLL_STATE_FILE):
        os.remove(ROLL_STATE_FILE)

if os.path.exists(DAILY_CACHE_FILE):
    with open(DAILY_CACHE_FILE) as f:
        cached_records = _json.load(f)
    cache_df = pd.DataFrame(cached_records)
    cache_df['Date'] = pd.to_datetime(cache_df['Date'])
    last_dates = cache_df.groupby('Market')['Date'].max()
    print(f"Loaded cache: {len(cache_df)} rows across {cache_df['Market'].nunique()} markets")
else:
    cache_df = pd.DataFrame()
    last_dates = pd.Series(dtype='datetime64[ns]')
    print("No cache found — will do a full back-adjusted backfill from 2022")

if os.path.exists(ROLL_STATE_FILE):
    with open(ROLL_STATE_FILE) as f:
        roll_state = _json.load(f)
else:
    roll_state = {}

# ── Connect to IB Gateway ─────────────────────────────────────────────
util.startLoop()
ib = IB()
ib.connect(IB_HOST, IB_PORT, clientId=CLIENT_ID)
print(f"Connected to IB Gateway: {ib.isConnected()}")


def _persist_cache(df, state):
    """Write cache and roll-state to disk atomically-ish (per-market checkpoint)."""
    save = df.copy()
    save['Date'] = save['Date'].dt.strftime('%Y-%m-%d')
    with open(DAILY_CACHE_FILE, 'w') as f:
        _json.dump(save.to_dict('records'), f)
    with open(ROLL_STATE_FILE, 'w') as f:
        _json.dump(state, f, default=str, indent=2)


# ── Per-market rebuild loop ──────────────────────────────────────────
failed_markets = []
today = date.today()
processed = 0

for market_name in unique_markets:
    row = mapping_df[mapping_df['Market'] == market_name]
    if row.empty:
        print(f"  ⚠ No IB mapping for {market_name}, skipping")
        failed_markets.append(market_name)
        continue

    display_sym   = row['IB_Symbol'].iloc[0]
    exchange      = row['IB_Exchange'].iloc[0]
    currency      = row['IB_Currency'].iloc[0]
    trading_class = row['IB_TradingClass'].iloc[0]

    # ---------- Determine which path to take for this market ----------------
    cache_market = (cache_df[cache_df['Market'] == market_name]
                    if not cache_df.empty else pd.DataFrame())
    if not cache_market.empty:
        last_cached_date = cache_market['Date'].max().date()
        last_row = cache_market.sort_values('Date').iloc[-1]
        last_cached_contract = last_row.get('Contract')
        if pd.isna(last_cached_contract):
            last_cached_contract = None
    else:
        last_cached_date = None
        last_cached_contract = None

    # State 2: cache is current through today -> skip
    if last_cached_date is not None and last_cached_date + timedelta(days=1) >= today:
        print(f"  ⏭ {market_name}: cache current through {last_cached_date}")
        continue

    # State 3: cached newest contract still front month -> incremental tail
    if last_cached_contract:
        front_contract, front_exp = find_liquid_front_contract(
            ib, future_symbol=display_sym, exchange=exchange,
            currency=currency, trading_class=trading_class,
        )
        if front_contract and front_contract.localSymbol == last_cached_contract:
            print(f"  {market_name} ({display_sym}): incremental from {last_cached_date}...", flush=True)
            try:
                tail = fetch_recent_bars(
                    ib, front_contract,
                    since_date=last_cached_date,
                    bar_size='1 day', what_to_show='TRADES', use_rth=False,
                )
            except Exception as e:
                print(f"     ✗ incremental error: {e}; falling through to full rebuild")
                tail = None

            if tail is not None:
                if not tail:
                    print(f"     ⏭ no new bars from {front_contract.localSymbol}")
                    processed += 1
                    time.sleep(1.5)
                    continue

                tail_rows = [{
                    'Date': pd.Timestamp(b.date),
                    'Open': b.open, 'High': b.high, 'Low': b.low,
                    'Close': b.close, 'Volume': b.volume,
                    'Market': market_name, 'IB_Symbol': display_sym,
                    'Contract': front_contract.localSymbol,
                } for b in tail]
                cache_df = pd.concat([cache_df, pd.DataFrame(tail_rows)],
                                     ignore_index=True)
                cache_df = cache_df.drop_duplicates(
                    subset=['Market', 'Date'], keep='last')
                cache_df = cache_df.sort_values(
                    ['Market', 'Date']).reset_index(drop=True)

                state = roll_state.get(market_name, {})
                state['last_incremental_at'] = datetime.now().isoformat(timespec='seconds')
                state['last_date'] = str(max(r['Date'].date() for r in tail_rows))
                roll_state[market_name] = state

                print(f"     ✓ +{len(tail_rows)} bars (incremental, no roll)")
                _persist_cache(cache_df, roll_state)
                processed += 1
                time.sleep(1.5)
                continue
        # else: front contract differs from cached -> a roll has crossed,
        # fall through to full rebuild for this market

    # State 1 or 4: full rebuild ----------------------------------------------
    if last_cached_contract:
        print(f"  {market_name} ({display_sym}): roll crossed -> full rebuild...", flush=True)
    else:
        print(f"  {market_name} ({display_sym}): full rebuild from scratch...", flush=True)

    try:
        bars, roll_log, status = build_backadjusted_continuous(
            ib,
            future_symbol=display_sym,
            exchange=exchange,
            currency=currency,
            trading_class=trading_class,
            start_date=START_DATE,
            bar_size='1 day',
            what_to_show='TRADES',
            use_rth=False,
            roll_buffer_business_days=ROLL_BUFFER_BDAYS,
        )
    except Exception as e:
        print(f"     ✗ builder error: {e}")
        failed_markets.append(market_name)
        continue

    if status != 'ok' or not bars:
        print(f"     ✗ {status} (no bars)")
        failed_markets.append(market_name)
        continue

    new_rows = [{
        'Date': pd.Timestamp(b['date']),
        'Open': b['open'], 'High': b['high'], 'Low': b['low'],
        'Close': b['close'], 'Volume': b['volume'],
        'Market': market_name, 'IB_Symbol': display_sym,
        'Contract': b['contract_local_symbol'],
    } for b in bars]
    new_df = pd.DataFrame(new_rows)

    if not cache_df.empty:
        cache_df = cache_df[cache_df['Market'] != market_name]
    cache_df = pd.concat([cache_df, new_df], ignore_index=True)
    cache_df = cache_df.sort_values(['Market', 'Date']).reset_index(drop=True)

    roll_state[market_name] = {
        'rebuilt_at': datetime.now().isoformat(timespec='seconds'),
        'first_date': str(new_df['Date'].min().date()),
        'last_date':  str(new_df['Date'].max().date()),
        'n_contracts': len(set(r['Contract'] for r in new_rows)),
        'rolls': [{
            'from': r['from'], 'to': r['to'],
            'nominal_roll': str(r.get('nominal_roll')),
            'actual_roll':  str(r.get('actual_roll')),
            'gap': r.get('gap', 0.0),
        } for r in roll_log],
    }

    print(f"     ✓ {len(new_rows)} bars, {len(roll_log)} rolls, "
          f"range {new_df['Date'].min().date()} → {new_df['Date'].max().date()}")

    _persist_cache(cache_df, roll_state)
    processed += 1
    time.sleep(1.5)

print(f"\nRebuilt {processed} markets")
if failed_markets:
    print(f"Failed / skipped: {failed_markets}")

# Restore Date as Timestamp for downstream cells
daily_price_df = cache_df.copy()
if not daily_price_df.empty and 'Date' in daily_price_df.columns:
    daily_price_df['Date'] = pd.to_datetime(daily_price_df['Date'])



Loaded cache: 34305 rows across 48 markets
Connected to IB Gateway: True
  ⏭ GOLD - COMMODITY EXCHANGE INC.: cache current through 2026-05-26
  ⏭ MICRO SILVER - COMMODITY EXCHANGE INC.: cache current through 2026-05-26
  ⏭ PLATINUM - NEW YORK MERCANTILE EXCHANGE: cache current through 2026-05-26
  ⏭ PALLADIUM - NEW YORK MERCANTILE EXCHANGE: cache current through 2026-05-26
  ⏭ COPPER- #1 - COMMODITY EXCHANGE INC.: cache current through 2026-05-26
  ⏭ MICRO COPPER - COMMODITY EXCHANGE INC.: cache current through 2026-05-26
  ⏭ SOYBEAN OIL - CHICAGO BOARD OF TRADE: cache current through 2026-05-26
  ⏭ NAT GAS NYME - NEW YORK MERCANTILE EXCHANGE: cache current through 2026-05-26
  ⏭ GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE: cache current through 2026-05-26
  ⏭ COCOA - ICE FUTURES U.S.: cache current through 2026-05-26
  ⏭ CORN - CHICAGO BOARD OF TRADE: cache current through 2026-05-26
  ⏭ OATS - CHICAGO BOARD OF TRADE: cache current through 2026-05-26
  ⏭ WHEAT-SRW - CHICAGO BOARD OF 

Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Error 1102, reqId -1: Connectivity between IBKR and Trader Workstation has been restored - data maintained. The following farms are connected: eufarm; secdefeu. The following farms are not connected: euhmds; ushmds.
Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Error 1102, reqId -1: Connectivity between IBKR and Trader Workstation has been restored - data maintained. The following farms are connected: eufarm. The following farms are not connected: euhmds; ushmds; secdefeu.
Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Error 1102, reqId -1: Connectivity between IBKR and Trader Workstation has been restored - data maintained. The following farms are connected: eufarm; secdefeu. The following farms are not connected: euhmds; ushmds.
Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Error 11

In [91]:
# The builder above already persisted cache + roll state inside its loop.
# This cell just prints a summary.

print(f"✓ Daily price cache: {len(daily_price_df)} records across {daily_price_df['Market'].nunique()} markets")
print(f"✓ Date range: {daily_price_df['Date'].min().date()} → {daily_price_df['Date'].max().date()}")
print(f"✓ Cache:        {DAILY_CACHE_FILE}")
print(f"✓ Roll state:   {ROLL_STATE_FILE}")

✓ Daily price cache: 34305 records across 48 markets
✓ Date range: 2023-05-26 → 2026-05-26
✓ Cache:        ib_daily_cache.json
✓ Roll state:   ib_daily_roll_state.json


In [92]:
print("=" * 80)
print("BACK-ADJUSTED CONTINUOUS-CONTRACT DIAGNOSTIC")
print("=" * 80)

target_markets = set(unique_markets)
price_markets = set(daily_price_df['Market'].unique())
missing_entirely = target_markets - price_markets

print(f"\nMarkets in unique_markets but NO price data ({len(missing_entirely)}):")
for m in sorted(missing_entirely):
    print(f"   - {m}")

df_filtered = daily_price_df[daily_price_df['Market'].isin(unique_markets)]
null_summary = df_filtered.groupby('Market').agg({
    'Close': lambda x: x.isnull().sum(),
    'Open':  lambda x: x.isnull().sum(),
    'High':  lambda x: x.isnull().sum(),
    'Low':   lambda x: x.isnull().sum(),
})
markets_with_nulls = null_summary[(null_summary > 0).any(axis=1)]
print(f"\nMarkets with NaNs in OHLC: {len(markets_with_nulls)}")
if len(markets_with_nulls):
    print(markets_with_nulls.to_string())

# Per-market sanity: max single-day return and roll summary
print(f"\nPer-market summary:")
print(f"  {'Market':<55} {'Bars':>6} {'Rolls':>5} {'MaxRtn%':>8} {'LastClose':>11}")
print(f"  {'-'*55} {'-'*6} {'-'*5} {'-'*8} {'-'*11}")
flagged = []
for m in sorted(target_markets & price_markets):
    sub = df_filtered[df_filtered['Market'] == m].sort_values('Date')
    if len(sub) < 2:
        continue
    rets = sub['Close'].pct_change().abs()
    max_rtn = float(rets.max() * 100) if not rets.empty else 0.0
    n_rolls = len(roll_state.get(m, {}).get('rolls', []))
    last_close = sub['Close'].iloc[-1]
    flag = ' ⚠' if max_rtn > 15 else ''
    print(f"  {m[:55]:<55} {len(sub):>6d} {n_rolls:>5d} {max_rtn:>8.2f} {last_close:>11.2f}{flag}")
    if max_rtn > 15:
        flagged.append((m, max_rtn))

if flagged:
    print(f"\n⚠ {len(flagged)} markets with >15% single-day moves on the adjusted series.")
    print("  Check these for residual roll discontinuities or bad bars.")

print(f"\nSummary:")
print(f"   unique_markets:         {len(target_markets)}")
print(f"   markets with data:      {len(price_markets & target_markets)}")
print(f"   markets missing:        {len(missing_entirely)}")
print(f"   roll-state entries:     {len(roll_state)}")
print("=" * 80)


BACK-ADJUSTED CONTINUOUS-CONTRACT DIAGNOSTIC

Markets in unique_markets but NO price data (1):
   - ERCOT.HB_HOUSTON_month_on_rtp - NODAL EXCHANGE

Markets with NaNs in OHLC: 0

Per-market summary:
  Market                                                    Bars Rolls  MaxRtn%   LastClose
  ------------------------------------------------------- ------ ----- -------- -----------
  AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE            752    24     5.01        0.72
  BITCOIN - CHICAGO MERCANTILE EXCHANGE                      638    24    13.13    77340.00
  BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE               764    24     4.16        0.20
  BRITISH POUND - CHICAGO MERCANTILE EXCHANGE                752    24     1.78        1.35
  CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE              751    24     1.84        0.72
  COCOA - ICE FUTURES U.S.                                   718    10    27.26     4152.00 ⚠
  COFFEE C - ICE FUTURES U.S.                                716

#### Add Price data to COT dataframe

In [93]:
df5 = pd.merge(df4, daily_price_df[['Market', 'Date', 'Close', 'Open', 'High', 'Low', 'Volume']], on=['Market', 'Date'], how='left')

##### df 5 now contains price and open interest by week but for all markets, not just the unique_markets

In [94]:
df5 = df5.sort_values(['Market', 'Date'], ascending = [True, True])

##### df6 has weekly COT and price data along with RSI but does not have daily prices

In [95]:
#Put the data above into a dataframe without the COT data for markets we are not trading (i.e removing everything not in unique markets)

df6 = []  # Initialize an empty list to store filtered dataframes

for market_name in unique_markets:
    df6.append(df5[df5["Market"] == market_name])

# Concatenate the filtered dataframes into a single DataFrame
df6 = pd.concat(df6, ignore_index=True)


        

In [96]:
# Calculate RSI for daily price data
daily_price_df['RSI'] = daily_price_df.groupby('Market')['Close'].transform(lambda x: rsi(x))

# Add data_type to existing df6 (weekly COT data)
df6_weekly = df6.copy()
df6_weekly['data_type'] = 'weekly_cot'

# Prepare daily price data to match df6 structure
daily_price_expanded = daily_price_df.copy()
daily_price_expanded['data_type'] = 'daily_price'

cot_columns = ['OI', 'OI_Index', 'Retail_Index', 'Commercial_Index', 'Traders_Index', 
               'Net Retail Position', 'Net Commercial Position', 'Net Traders Position',
               'Noncommercial Positions-Long (All)', 'Noncommercial Positions-Short (All)',
               'Commercial Positions-Long (All)', 'Commercial Positions-Short (All)',
               'Nonreportable Positions-Long (All)', 'Nonreportable Positions-Short (All)']

for col in cot_columns:
    if col not in daily_price_expanded.columns:
        daily_price_expanded[col] = None

if 'group' not in daily_price_expanded.columns:
    daily_price_expanded = pd.merge(daily_price_expanded, mapping_df[['Market', 'group']], on='Market', how='left')

if 'IB_Symbol' not in df6_weekly.columns:
    df6_weekly = pd.merge(df6_weekly, mapping_df[['Market', 'IB_Symbol']], on='Market', how='left')

common_columns = ['Market', 'Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'RSI', 'IB_Symbol', 'data_type', 'group'] + cot_columns

df6_weekly = df6_weekly.reindex(columns=common_columns, fill_value=None)
daily_price_expanded = daily_price_expanded.reindex(columns=common_columns, fill_value=None)

print(f"✓ Common columns: {common_columns[:12]}...")

df6_combined = pd.concat([df6_weekly, daily_price_expanded], ignore_index=True)
df6_combined = df6_combined.sort_values(['Market', 'Date']).reset_index(drop=True)

print(f"Combined dataset: {len(df6_combined)} total records")
print(f"  Weekly COT: {len(df6_weekly)} | Daily price: {len(daily_price_expanded)}")


✓ Common columns: ['Market', 'Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'RSI', 'IB_Symbol', 'data_type', 'group', 'OI']...
Combined dataset: 43642 total records
  Weekly COT: 9337 | Daily price: 34305


/var/folders/km/x_cb0wn5249ddhfzzt066kg40000gn/T/ipykernel_82930/1487681820.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df6_combined = pd.concat([df6_weekly, daily_price_expanded], ignore_index=True)


In [97]:
df6_combined = df6_combined.dropna(subset=['Close'])

In [98]:
# Filter for March 9th, 2026 only
df_march_9 = df6_combined[df6_combined['Date'] == '2026-03-10']

markets_with_close = df_march_9[df_march_9['Close'].notna()]['Market'].unique()

print(f"Total markets with close prices on March 10th: {len(markets_with_close)}")
print("\nMarkets with close prices on March 10th:")
for market in sorted(markets_with_close):
    print(f"  - {market}")

Total markets with close prices on March 10th: 48

Markets with close prices on March 10th:
  - AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE
  - BITCOIN - CHICAGO MERCANTILE EXCHANGE
  - BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE
  - BRITISH POUND - CHICAGO MERCANTILE EXCHANGE
  - CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE
  - COCOA - ICE FUTURES U.S.
  - COFFEE C - ICE FUTURES U.S.
  - COPPER- #1 - COMMODITY EXCHANGE INC.
  - CORN - CHICAGO BOARD OF TRADE
  - COTTON NO. 2 - ICE FUTURES U.S.
  - E-MINI NATURAL GAS - NEW YORK MERCANTILE EXCHANGE
  - E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE
  - EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE
  - EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE
  - FEEDER CATTLE - CHICAGO MERCANTILE EXCHANGE
  - GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE
  - GOLD - COMMODITY EXCHANGE INC.
  - JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE
  - LEAN HOGS - CHICAGO MERCANTILE EXCHANGE
  - LIVE CATTLE - CHICAGO MERCANTILE EXCHANGE
 

In [99]:
platinum = df6_combined[df6_combined["Market"].str.contains("COCOA")]
platinum.sort_values("Date", ascending = False)

,Market,Date,Close,Open,High,Low,Volume,RSI,IB_Symbol,data_type,...,Traders_Index,Net Retail Position,Net Commercial Position,Net Traders Position,Noncommercial Positions-Long (All),Noncommercial Positions-Short (All),Commercial Positions-Long (All),Commercial Positions-Short (All),Nonreportable Positions-Long (All),Nonreportable Positions-Short (All)
5743,COCOA - ICE FUTURES U.S.,2026-05-26,4152.0,3790.0,4199.0,3765.0,15097.0,32.135985,CC,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
5742,COCOA - ICE FUTURES U.S.,2026-05-22,3796.0,3735.0,3829.0,3651.0,9232.0,38.843931,CC,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
5741,COCOA - ICE FUTURES U.S.,2026-05-21,3767.0,3832.0,3943.0,3732.0,10590.0,33.042138,CC,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
5740,COCOA - ICE FUTURES U.S.,2026-05-20,3889.0,3846.0,3938.0,3763.0,8696.0,44.187146,CC,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
5739,COCOA - ICE FUTURES U.S.,2026-05-19,3907.0,3769.0,3958.0,3769.0,10060.0,46.132469,CC,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,COCOA - ICE FUTURES U.S.,2023-07-25,826.0,775.0,837.0,775.0,214.0,NaN,CC,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
4881,COCOA - ICE FUTURES U.S.,2023-07-24,775.0,754.0,776.0,749.0,111.0,NaN,CC,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
4880,COCOA - ICE FUTURES U.S.,2023-07-21,764.0,727.0,772.0,727.0,192.0,NaN,CC,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
4879,COCOA - ICE FUTURES U.S.,2023-07-20,738.0,751.0,752.0,725.0,288.0,NaN,CC,daily_price,...,NaN,None,None,None,None,None,None,None,None,None


In [100]:
# Export the combined dataset (both weekly COT data and daily price data)
df6_combined.to_json('cot_data.json', orient='records')
print("Combined data (weekly COT + daily prices) exported to cot_data.json")
print(f"Total records exported: {len(df6_combined)}")

Combined data (weekly COT + daily prices) exported to cot_data.json
Total records exported: 40262


In [101]:
# =============================================================================
# 🔔 CHECK FOR RECENT TRADING SIGNALS (Last 7 Days)
# =============================================================================
from datetime import datetime, timedelta

def check_recent_signals(df, days=7, commercial_long=80, commercial_short=20, 
                         rsi_oversold=30, rsi_overbought=70):
    """Check for trading signals in the last N days."""
    cutoff_date = datetime.now() - timedelta(days=days)
    recent_signals = []
    
    markets = df['Market'].unique()
    
    for market in markets:
        market_data = df[df['Market'] == market].copy()
        price_daily = market_data[market_data['data_type'] == 'daily_price'].copy()
        cot_weekly = market_data[market_data['data_type'] == 'weekly_cot'].copy()
        
        if price_daily.empty or cot_weekly.empty:
            continue
        
        # Check if Commercial_Index exists in COT data
        if 'Commercial_Index' not in cot_weekly.columns:
            continue
            
        # Sort both by Date for merge_asof
        price_daily = price_daily[['Date', 'Close', 'RSI']].copy().sort_values('Date').reset_index(drop=True)
        cot_for_merge = cot_weekly[['Date', 'Commercial_Index']].dropna(subset=['Commercial_Index']).copy()
        cot_for_merge = cot_for_merge.sort_values('Date').reset_index(drop=True)
        
        if cot_for_merge.empty:
            continue
        
        # Use merge_asof to carry forward most recent COT data to each daily price row
        # direction='backward' means: find most recent COT date <= each price date
        merged = pd.merge_asof(price_daily, cot_for_merge, on='Date', direction='backward')
        
        # Drop rows with missing required data
        merged = merged.dropna(subset=['Close', 'RSI', 'Commercial_Index'])
        
        # Filter to recent days
        merged = merged[merged['Date'] >= cutoff_date]
        
        for _, row in merged.iterrows():
            signal_type = None
            # Long: Commercial >= 80 AND RSI < 30
            if row['Commercial_Index'] >= commercial_long and row['RSI'] < rsi_oversold:
                signal_type = 'LONG'
            # Short: Commercial <= 20 AND RSI > 70
            elif row['Commercial_Index'] <= commercial_short and row['RSI'] > rsi_overbought:
                signal_type = 'SHORT'
            
            if signal_type:
                recent_signals.append({
                    'Market': market,
                    'Date': row['Date'],
                    'Signal': signal_type,
                    'COT': round(row['Commercial_Index'], 1),
                    'RSI': round(row['RSI'], 1),
                    'Close': round(row['Close'], 2)
                })
    
    return sorted(recent_signals, key=lambda x: x['Date'], reverse=True)

# Check for signals
signals = check_recent_signals(df6_combined, days=7)

print("=" * 70)
if signals:
    print(f"🔔 ALERT: {len(signals)} TRADING SIGNAL(S) IN THE LAST 7 DAYS!")
    print("=" * 70)
    for sig in signals:
        emoji = "🟢" if sig['Signal'] == 'LONG' else "🔴"
        print(f"{emoji} {sig['Signal']:5} | {sig['Date'].strftime('%Y-%m-%d')} | {sig['Market'][:40]}")
        print(f"         COT: {sig['COT']}, RSI: {sig['RSI']}, Price: ${sig['Close']:,.2f}")
        print("-" * 70)
else:
    print("✓ No new trading signals in the last 7 days")
    print("=" * 70)


🔔 ALERT: 8 TRADING SIGNAL(S) IN THE LAST 7 DAYS!
🟢 LONG  | 2026-05-26 | MICRO ETHER - CHICAGO MERCANTILE EXCHANG
         COT: 84.0, RSI: 24.8, Price: $2,106.00
----------------------------------------------------------------------
🟢 LONG  | 2026-05-26 | MICRO GOLD - COMMODITY EXCHANGE INC.
         COT: 94.0, RSI: 25.8, Price: $4,561.10
----------------------------------------------------------------------
🟢 LONG  | 2026-05-22 | MICRO ETHER - CHICAGO MERCANTILE EXCHANG
         COT: 84.0, RSI: 21.1, Price: $2,062.00
----------------------------------------------------------------------
🟢 LONG  | 2026-05-22 | MICRO GOLD - COMMODITY EXCHANGE INC.
         COT: 94.0, RSI: 16.5, Price: $4,523.20
----------------------------------------------------------------------
🟢 LONG  | 2026-05-22 | UST 2Y NOTE - CHICAGO BOARD OF TRADE
         COT: 89.7, RSI: 21.6, Price: $103.12
----------------------------------------------------------------------
🟢 LONG  | 2026-05-21 | MICRO ETHER - CHICAGO MERCA

In [102]:
df6_combined.iloc[-1]

Market                                 WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE
Date                                                           2026-05-26 00:00:00
Close                                                                        91.97
Open                                                                         93.88
High                                                                          93.9
Low                                                                          90.32
Volume                                                                     44770.0
RSI                                                                      45.378471
IB_Symbol                                                                       CL
data_type                                                              daily_price
group                                                                     Energies
OI                                                                            None
OI_I

## Build Intraday Cache from IB (30m / 60m)

Populates `ORB_intraday_data.json` so that `Inside_day_backtest.py` and `ORB_backtest.py` can read from cache instead of calling Yahoo Finance.

IB limits: 30m data goes back ~30-60 days only. Run this cell periodically to keep the cache fresh.

In [103]:
# Build ORB_intraday_data.json using back-adjusted continuous 30m / 60m series.
# IB's 30m/1h history goes back ~6 months, so most markets show 1-2 rolls;
# the same Panama back-adjustment as the daily series is applied so backtest
# and live use the same price scale.

INTRADAY_CACHE_FILE = 'ORB_intraday_data.json'
INTRADAY_STATE_FILE = 'ORB_intraday_roll_state.json'

# Force a full intraday rebuild (set False on routine runs to skip already-current
# (symbol, interval) pairs). Required True the first time after switching to
# back-adjusted because the prior cache was built on raw ContFuture data.
FORCE_FULL_INTRADAY_REBUILD = False

if FORCE_FULL_INTRADAY_REBUILD and os.path.exists(INTRADAY_CACHE_FILE):
    print("FORCE_FULL_INTRADAY_REBUILD=True → wiping intraday cache.")
    os.remove(INTRADAY_CACHE_FILE)
    if os.path.exists(INTRADAY_STATE_FILE):
        os.remove(INTRADAY_STATE_FILE)

if os.path.exists(INTRADAY_CACHE_FILE):
    with open(INTRADAY_CACHE_FILE) as f:
        existing = _json.load(f)
    intra_cache = pd.DataFrame(existing)
    if not intra_cache.empty:
        intra_cache['datetime'] = pd.to_datetime(
            intra_cache['datetime'], utc=True).dt.tz_localize(None)
    print(f"Loaded intraday cache: {len(intra_cache)} rows")
else:
    intra_cache = pd.DataFrame()
    print("No intraday cache — building from scratch")

intra_state = {}
if os.path.exists(INTRADAY_STATE_FILE):
    with open(INTRADAY_STATE_FILE) as f:
        intra_state = _json.load(f)


def _persist_intraday(df, state):
    save = df.copy()
    save['datetime'] = save['datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
    with open(INTRADAY_CACHE_FILE, 'w') as f:
        _json.dump(save.to_dict('records'), f)
    with open(INTRADAY_STATE_FILE, 'w') as f:
        _json.dump(state, f, default=str, indent=2)


INTRADAY_FRESH_HOURS = 4  # Skip (symbol, interval) if last cached bar is newer than this.

for market_name in unique_markets:
    row = mapping_df[mapping_df['Market'] == market_name]
    if row.empty:
        continue

    display_sym   = row['IB_Symbol'].iloc[0]
    exchange      = row['IB_Exchange'].iloc[0]
    currency      = row['IB_Currency'].iloc[0]
    trading_class = row['IB_TradingClass'].iloc[0]

    for interval in ['30m', '60m']:
        bar_size = '30 mins' if interval == '30m' else '1 hour'

        # Determine cache state for this (market, interval)
        last_cached_dt = None
        last_cached_contract = None
        if not intra_cache.empty:
            mask = (intra_cache['symbol'] == market_name) & (intra_cache['interval'] == interval)
            sub = intra_cache.loc[mask]
            if not sub.empty:
                last_cached_dt = sub['datetime'].max()
                last_row = sub.sort_values('datetime').iloc[-1]
                if 'contract' in sub.columns:
                    val = last_row.get('contract')
                    if not pd.isna(val):
                        last_cached_contract = val

        # State 2: cache fresh enough -> skip
        if last_cached_dt is not None and \
                ((pd.Timestamp.now() - last_cached_dt).days + 1) <= 1:
            print(f"  ⏭ {market_name} {interval}: fresh ({last_cached_dt})")
            continue

        # State 3: cached newest contract still front month -> incremental tail
        if last_cached_contract:
            front_contract, _ = find_liquid_front_contract(
                ib, future_symbol=display_sym, exchange=exchange,
                currency=currency, trading_class=trading_class,
            )
            if front_contract and front_contract.localSymbol == last_cached_contract:
                print(f"  {market_name} {interval}: incremental from {last_cached_dt}...", flush=True)
                since_d = last_cached_dt.date() if hasattr(last_cached_dt, 'date') else last_cached_dt
                try:
                    tail = fetch_recent_bars(
                        ib, front_contract,
                        since_date=since_d,
                        bar_size=bar_size, what_to_show='TRADES', use_rth=False,
                    )
                except Exception as e:
                    print(f"     ✗ incremental error: {e}; falling through to full rebuild")
                    tail = None

                if tail is not None:
                    # Drop intraday bars whose timestamp <= last cached datetime
                    # (normalize tz — cache is naive UTC-stripped, IB bars are often tz-aware)
                    _lc = pd.Timestamp(last_cached_dt)
                    if _lc.tzinfo is not None:
                        _lc = _lc.tz_convert('UTC').tz_localize(None)
                    new_only = []
                    for b in tail:
                        bt = pd.Timestamp(b.date)
                        if bt.tzinfo is not None:
                            bt = bt.tz_convert('UTC').tz_localize(None)
                        if bt > _lc:
                            new_only.append(b)
                    if not new_only:
                        print(f"     ⏭ no new bars from {front_contract.localSymbol}")
                        time.sleep(1.5)
                        continue
                    tail_rows = [{
                        'symbol':   market_name,
                        'interval': interval,
                        'datetime': pd.Timestamp(b.date),
                        'open':  b.open,  'high': b.high,
                        'low':   b.low,   'close': b.close,
                        'volume': b.volume,
                        'contract': front_contract.localSymbol,
                    } for b in new_only]
                    tail_df = pd.DataFrame(tail_rows)
                    tail_df['datetime'] = pd.to_datetime(
                        tail_df['datetime'], utc=True).dt.tz_localize(None)
                    intra_cache = pd.concat([intra_cache, tail_df], ignore_index=True)
                    intra_cache = intra_cache.drop_duplicates(
                        subset=['symbol', 'interval', 'datetime'], keep='last')
                    intra_cache = intra_cache.sort_values(
                        ['symbol', 'interval', 'datetime']).reset_index(drop=True)

                    state = intra_state.get(f'{market_name}|{interval}', {})
                    state['last_incremental_at'] = datetime.now().isoformat(timespec='seconds')
                    state['last'] = str(tail_df['datetime'].max())
                    intra_state[f'{market_name}|{interval}'] = state

                    print(f"     ✓ +{len(tail_rows)} bars (incremental, no roll)")
                    _persist_intraday(intra_cache, intra_state)
                    time.sleep(1.5)
                    continue
            # else: roll has crossed -> full rebuild

        # State 1 or 4: full rebuild for this (market, interval)
        if last_cached_contract:
            print(f"  {market_name} {interval}: roll crossed -> full rebuild...", flush=True)
        else:
            print(f"  {market_name} {interval} ({display_sym}): full rebuild...", flush=True)

        try:
            bars, roll_log, status = build_backadjusted_continuous(
                ib,
                future_symbol=display_sym,
                exchange=exchange,
                currency=currency,
                trading_class=trading_class,
                start_date=date.today() - timedelta(days=60),
                bar_size=bar_size,
                what_to_show='TRADES',
                use_rth=False,
                roll_buffer_business_days=ROLL_BUFFER_BDAYS,
            )
        except Exception as e:
            print(f"     ✗ builder error: {e}")
            continue

        if status != 'ok' or not bars:
            print(f"     ✗ {status}")
            continue

        new_rows = [{
            'symbol':   market_name,
            'interval': interval,
            'datetime': pd.Timestamp(b['date']),
            'open':  b['open'],  'high': b['high'],
            'low':   b['low'],   'close': b['close'],
            'volume': b['volume'],
            'contract': b['contract_local_symbol'],
        } for b in bars]
        new_df = pd.DataFrame(new_rows)
        new_df['datetime'] = pd.to_datetime(
            new_df['datetime'], utc=True).dt.tz_localize(None)

        if not intra_cache.empty:
            keep_mask = ~((intra_cache['symbol'] == market_name) &
                          (intra_cache['interval'] == interval))
            intra_cache = intra_cache[keep_mask]
        intra_cache = pd.concat([intra_cache, new_df], ignore_index=True)
        intra_cache = intra_cache.sort_values(
            ['symbol', 'interval', 'datetime']).reset_index(drop=True)

        intra_state[f'{market_name}|{interval}'] = {
            'rebuilt_at': datetime.now().isoformat(timespec='seconds'),
            'first': str(new_df['datetime'].min()),
            'last':  str(new_df['datetime'].max()),
            'n_contracts': len(set(r['contract'] for r in new_rows)),
            'n_rolls': len(roll_log),
        }
        print(f"     ✓ {len(new_rows)} bars, {len(roll_log)} rolls")

        _persist_intraday(intra_cache, intra_state)
        time.sleep(1.5)

print(f"\n✓ Intraday cache: {len(intra_cache)} total rows → {INTRADAY_CACHE_FILE}")


Loaded intraday cache: 90515 rows
  ⏭ GOLD - COMMODITY EXCHANGE INC. 30m: fresh (2026-05-26 20:30:00)
  ⏭ GOLD - COMMODITY EXCHANGE INC. 60m: fresh (2026-05-26 20:00:00)
  ⏭ MICRO SILVER - COMMODITY EXCHANGE INC. 30m: fresh (2026-05-26 20:30:00)
  ⏭ MICRO SILVER - COMMODITY EXCHANGE INC. 60m: fresh (2026-05-26 20:00:00)
  ⏭ PLATINUM - NEW YORK MERCANTILE EXCHANGE 30m: fresh (2026-05-26 20:30:00)
  ⏭ PLATINUM - NEW YORK MERCANTILE EXCHANGE 60m: fresh (2026-05-26 20:00:00)
  ⏭ PALLADIUM - NEW YORK MERCANTILE EXCHANGE 30m: fresh (2026-05-26 20:30:00)
  ⏭ PALLADIUM - NEW YORK MERCANTILE EXCHANGE 60m: fresh (2026-05-26 20:00:00)
  ⏭ COPPER- #1 - COMMODITY EXCHANGE INC. 30m: fresh (2026-05-26 20:30:00)
  ⏭ COPPER- #1 - COMMODITY EXCHANGE INC. 60m: fresh (2026-05-26 20:00:00)
  ⏭ MICRO COPPER - COMMODITY EXCHANGE INC. 30m: fresh (2026-05-26 20:30:00)
  ⏭ MICRO COPPER - COMMODITY EXCHANGE INC. 60m: fresh (2026-05-26 20:00:00)
  ⏭ SOYBEAN OIL - CHICAGO BOARD OF TRADE 30m: fresh (2026-05-26 18:0

In [105]:
ib.disconnect()
print("Disconnected from IB Gateway")

Disconnected from IB Gateway


In [59]:
import pandas as pd
df = pd.read_json("ib_daily_cache.json")
m = "E-MINI NATURAL GAS - NEW YORK MERCANTILE EXCHANGE"
print(df[df["Market"]==m].tail(3)[["Date","Contract","Close"]])

           Date Contract  Close
7934 2026-05-11     QGM6  2.930
7935 2026-05-12     QGM6  2.835
7936 2026-05-13     QGM6  2.865
